In [15]:
import numpy as np
import pandas as pd
from transformers import BertModel, BertTokenizer

In [16]:
df = pd.read_csv(
    '/kaggle/input/datasets/nourayman21k/arabic-dataset-hotels/unbalanced-reviews.txt',
    sep='\t',
    encoding='utf-16-le',
    on_bad_lines='skip'
)

In [17]:
df.shape

(409562, 7)

In [18]:
df.isna().sum()

no            0
Hotel name    0
rating        0
user type     0
room type     0
nights        0
review        0
dtype: int64

In [19]:
df = df.dropna(subset=['review', 'rating'])
df['review'] = df['review'].astype(str).str.strip()

In [20]:
df = df[df['review'].str.len() > 2]
df = df.drop_duplicates(subset=['review'])

In [21]:
label_map = {1: 0, 2: 0, 3: 1, 4: 2, 5: 2}
df['label'] = df['rating'].map(label_map)

In [22]:
df = df[['review', 'label']].reset_index(drop=True)

In [23]:
print(f"Final dataset size: {len(df):,}")

Final dataset size: 400,081


In [24]:
label_names = {0: 'negative', 1: 'neutral', 2: 'positive'}
for lbl, name in label_names.items():
    n = (df['label'] == lbl).sum()
    print(f"  {name:10s}  →  {n:,}  ({n/len(df)*100:.1f}%)")

print("\nSample:")
print(df.head(3))

  negative    →  52,167  (13.0%)
  neutral     →  79,518  (19.9%)
  positive    →  268,396  (67.1%)

Sample:
                                              review  label
0  “فندق راقي وسوف تتكرر زيارتي له”. الفندق بصراح...      2
1                   “ممتاز”. النظافة والطاقم متعاون.      0
2  استثنائي. سهولة إنهاء المعاملة في الاستقبال. ل...      2


In [44]:
df_small = df.sample(n=250000, random_state=42)

In [47]:
df_small.shape

(250000, 2)

In [48]:
from sklearn.model_selection import train_test_split
train_df , temp_df = train_test_split(df_small, test_size=0.2,stratify=df_small['label'],random_state=42)
val_df, test_df = train_test_split(temp_df , test_size=0.5,stratify=temp_df['label'],random_state=42)

In [49]:
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

In [50]:
print(f"Train : {len(train_df):,}")
print(f"Val   : {len(val_df):,}")
print(f"Test  : {len(test_df):,}")

label_names = {0: 'neg', 1: 'neu', 2: 'pos'}

for split_name, split in [('train', train_df), ('val', val_df), ('test', test_df)]:
    ratios = split['label'].value_counts(normalize=True).sort_index()
    parts  = '  '.join(f"{label_names[l]}: {ratios[l]*100:.1f}%" for l in [0,1,2])
    print(f"{split_name:6s} → {parts}")

Train : 200,000
Val   : 25,000
Test  : 25,000
train  → neg: 13.0%  neu: 19.9%  pos: 67.1%
val    → neg: 13.0%  neu: 19.8%  pos: 67.1%
test   → neg: 13.0%  neu: 19.9%  pos: 67.1%


In [51]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('aubmindlab/bert-base-arabertv02')

In [52]:
sample = "الفندق نظيف جداً والخدمة ممتازة"

tokens = tokenizer(sample, max_length=128, padding='max_length', truncation=True,return_tensors='pt')
print(tokens['input_ids'])


tensor([[    2, 11219,  8980,     1, 36258, 18074,     3,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,  

In [53]:
sample_reviews = train_df['review'][:5].tolist()

batch = tokenizer(
    sample_reviews,
    max_length=128,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

print(batch['input_ids'].shape)
print(batch['attention_mask'].shape)
print(batch['token_type_ids'].shape)


print("\nFirst review:", sample_reviews[0][:60])
real_token_count = batch['attention_mask'][0].sum().item()
print(f"Real tokens (non-padding): {real_token_count} / 128")


torch.Size([5, 128])
torch.Size([5, 128])
torch.Size([5, 128])

First review: “فندق جيد بمكان شعبي و لكنه جميل للتسوق”. تفاعل الفندق مع مش
Real tokens (non-padding): 36 / 128


In [54]:
import torch
from torch.utils.data import Dataset

class ReviewDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df['review'].tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'token_type_ids': encoding['token_type_ids'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [55]:
train_dataset = ReviewDataset(train_df, tokenizer)
val_dataset   = ReviewDataset(val_df, tokenizer)
test_dataset  = ReviewDataset(test_df, tokenizer)

In [56]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=train_df['label'].values
)

In [57]:
weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", {0: f"{class_weights[0]:.2f}",
                         1: f"{class_weights[1]:.2f}",
                         2: f"{class_weights[2]:.2f}"})

Class weights: {0: '2.56', 1: '1.68', 2: '0.50'}


In [59]:
from transformers import Trainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fn = torch.nn.CrossEntropyLoss(
            weight=weights_tensor.to(logits.device)
        )
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [60]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

In [61]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    'aubmindlab/bert-base-arabertv02',
    num_labels=3
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

In [62]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(device)

cuda


In [63]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',

    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,

    eval_strategy='epoch',
    save_strategy='epoch',
    logging_dir='./logs',
    logging_steps=200,

    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,

    fp16=torch.cuda.is_available(),
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [64]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        'f1':       f1_score(labels, preds, average='weighted'),
        'accuracy': accuracy_score(labels, preds),
        'f1_negative': f1_score(labels, preds, average=None)[0],
        'f1_neutral':  f1_score(labels, preds, average=None)[1],
        'f1_positive': f1_score(labels, preds, average=None)[2],
    }

In [66]:
from transformers import Trainer

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

In [67]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1,Accuracy,F1 Negative,F1 Neutral,F1 Positive
1,0.329195,0.337038,0.883852,0.878440,0.837020,0.744593,0.934108
2,0.318263,0.333985,0.885681,0.881480,0.833898,0.741007,0.938498
3,0.268676,0.349789,0.892216,0.889400,0.835485,0.752743,0.944455


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=18750, training_loss=0.3341395782470703, metrics={'train_runtime': 9748.4245, 'train_samples_per_second': 61.548, 'train_steps_per_second': 1.923, 'total_flos': 3.94670126592e+16, 'train_loss': 0.3341395782470703, 'epoch': 3.0})

In [70]:
trainer.evaluate(test_dataset)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.35100772976875305,
 'eval_f1': 0.8914441217944277,
 'eval_accuracy': 0.8886,
 'eval_f1_negative': 0.8345150098054005,
 'eval_f1_neutral': 0.7538882081251768,
 'eval_f1_positive': 0.9431658628899334,
 'eval_runtime': 125.2757,
 'eval_samples_per_second': 199.56,
 'eval_steps_per_second': 3.121,
 'epoch': 3.0}

In [71]:
import os

SAVE_DIR = "/kaggle/working/arabert_sentiment_model"

# 1. Create directory
os.makedirs(SAVE_DIR, exist_ok=True)

# 2. Save model (weights + config)
trainer.model.save_pretrained(SAVE_DIR)

# 3. Save tokenizer (VERY IMPORTANT)
tokenizer.save_pretrained(SAVE_DIR)

# 4. (Optional but recommended) Save training args
trainer.save_state()

# 5. Check saved files
print("Saved files:")
for file in os.listdir(SAVE_DIR):
    print(file)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved files:
config.json
model.safetensors
tokenizer.json
tokenizer_config.json


In [72]:
import shutil
import os

# Zip the entire working directory
shutil.make_archive('/kaggle/working/all_outputs', 'zip', '/kaggle/working')
print("Done! Download all_outputs.zip")

Done! Download all_outputs.zip
